In [65]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import MinMaxScaler

In [66]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [67]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [68]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

## Test dataset: MAASTRO 

In [69]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [70]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [71]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [72]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [73]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [74]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [75]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# Set y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [76]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [77]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [78]:
# Change the name of a column 'DFS_event' in the clincial_test 
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [79]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

### COX PLSR 

In [80]:
# Choose features from the result of Cox PLSR in R
plsr = [
"shape_MajorAxisLength",
"LBP_120_PET",
"LBP_201_PET",
"glszm_SmallAreaLowGrayLevelEmphasis_CT_c16",
"shape_Maximum3DDiameter",
"shape_SurfaceVolumeRatio",
"shape_Sphericity",
"shape_Elongation"
] 

In [81]:
X_plsr = X.loc[:, plsr]
X_new = X_plsr.copy()

In [82]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, plsr]

# Standardization

In [83]:
# Copy the original X for later 
original_X = X.copy()

In [84]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = MinMaxScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [85]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [86]:
X_new

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation
0,42.073251,0.140311,0.000062,0.029425,47.339202,0.251218,0.761164,0.600926
1,24.613845,0.191058,0.000349,0.037915,28.106939,0.489853,0.697049,0.841579
2,48.030294,0.126531,0.000000,0.008009,60.049979,0.278467,0.565792,0.772821
3,25.589900,0.192388,0.000000,0.018398,32.572995,0.474018,0.684364,0.847727
4,34.684750,0.202073,0.000399,0.013051,39.962482,0.563135,0.503142,0.831483
...,...,...,...,...,...,...,...,...
134,33.069705,0.152626,0.000000,0.014938,37.696154,0.322021,0.742102,0.680294
135,41.043692,0.142778,0.000079,0.011441,52.430907,0.227705,0.722918,0.758193
136,36.618802,0.140582,0.000000,0.020431,42.743421,0.298398,0.652963,0.770113
137,45.870392,0.156640,0.000054,0.019663,51.536395,0.252893,0.724255,0.628897


In [87]:
X_new_std

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation
0,0.350904,0.174091,0.088609,0.603741,0.298726,0.263957,0.828413,0.461898
1,0.123069,0.456736,0.501821,0.800232,0.099559,0.707641,0.645006,0.829873
2,0.428640,0.097341,0.000000,0.108112,0.430358,0.314620,0.269537,0.724738
3,0.135806,0.464144,0.000000,0.348551,0.145809,0.678200,0.608721,0.839273
4,0.254489,0.518081,0.572624,0.224799,0.222333,0.843892,0.090323,0.814436
...,...,...,...,...,...,...,...,...
134,0.233413,0.242682,0.000000,0.268474,0.198864,0.395598,0.773882,0.583257
135,0.337469,0.187835,0.113114,0.187545,0.351455,0.220239,0.719008,0.702369
136,0.279727,0.175602,0.000000,0.395599,0.251133,0.351677,0.518897,0.720597
137,0.400455,0.265036,0.077871,0.377829,0.342192,0.267071,0.722831,0.504667


In [88]:
MAASTRO_new 

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation
0,50.002093,0.122209,0.000026,0.010495,58.864251,0.215184,0.668072,0.765178
1,41.753334,0.128976,0.000167,0.035018,48.723711,0.276092,0.669961,0.776540
2,44.375483,0.137282,0.000000,0.012819,48.969378,0.298887,0.624081,0.697164
3,46.115989,0.171595,0.000080,0.029974,55.226805,0.361096,0.577624,0.574636
4,54.394967,0.128134,0.000035,0.013458,67.089492,0.251519,0.630933,0.633419
...,...,...,...,...,...,...,...,...
94,34.218615,0.137194,0.000078,0.039092,43.520110,0.307574,0.671754,0.882411
95,51.046869,0.137620,0.000054,0.015831,52.440442,0.289922,0.632189,0.535802
96,50.417953,0.115099,0.000028,0.011031,57.671483,0.228184,0.645548,0.716610
97,44.901412,0.117654,0.000000,0.019801,51.478151,0.223872,0.727488,0.665145


In [89]:
MAASTRO_new_std

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation
0,0.454371,0.073271,0.037694,0.165637,0.418078,0.196961,0.562115,0.713050
1,0.346729,0.110963,0.240548,0.733175,0.313064,0.310203,0.567520,0.730423
2,0.380947,0.157221,0.000000,0.219419,0.315608,0.352585,0.436277,0.609052
3,0.403660,0.348334,0.114827,0.616439,0.380409,0.468248,0.303383,0.421699
4,0.511695,0.106269,0.050658,0.234227,0.503258,0.264517,0.455879,0.511582
...,...,...,...,...,...,...,...,...
94,0.248406,0.156731,0.111676,0.827475,0.259176,0.368738,0.572648,0.892307
95,0.468005,0.159105,0.078027,0.289124,0.351554,0.335917,0.459472,0.362320
96,0.459798,0.033672,0.040540,0.178051,0.405726,0.221130,0.497683,0.638788
97,0.387810,0.047900,0.000000,0.381019,0.341589,0.213114,0.732079,0.560093


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [90]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 09:46:35,576] A new study created in memory with name: no-name-f79f46ba-f3bb-4d16-86fe-c616f737d22b


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-16 09:46:35,910] A new study created in memory with name: no-name-6803fa01-3973-4815-99a6-56f11f9c1a7c


Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.7081545064377682
[I 2024-04-16 09:46:35,904] Trial 0 finished with value: 0.6811103040058727 and parameters: {}. Best is trial 0 with value: 0.6811103040058727.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6811103040058727], datetime_start=datetime.datetime(2024, 4, 16, 9, 46, 35, 631107), datetime_complete=datetime.datetime(2024, 4, 16, 9, 46, 35, 904392), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6811103040058727


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2644313749672424
Fold 2 IBS: 0.1772296022395614
Fold 3 IBS: 0.19936774967703333
Fold 4 IBS: 0.24814797488840457
Fold 5 IBS: 0.19174971398784724
[I 2024-04-16 09:46:36,189] Trial 0 finished with value: 0.21618528315201777 and parameters: {}. Best is trial 0 with value: 0.21618528315201777.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21618528315201777], datetime_start=datetime.datetime(2024, 4, 16, 9, 46, 35, 948840), datetime_complete=datetime.datetime(2024, 4, 16, 9, 46, 36, 189248), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21618528315201777


In [91]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [92]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.681
train_ibs:  0.216


#### Test

In [93]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [94]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.508
IBS score: 0.283


In [95]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [96]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [97]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 09:46:36,341] A new study created in memory with name: no-name-3038044e-8414-4e98-82eb-922a00b56e15


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5876494023904383
Fold 2 C-index: 0.7073643410852714
Fold 3 C-index: 0.5446808510638298
Fold 4 C-index: 0.6711026615969582
Fold 5 C-index: 0.6437768240343348
[I 2024-04-16 09:46:36,473] Trial 0 finished with value: 0.6309148160341665 and parameters: {}. Best is trial 0 with value: 0.6309148160341665.


[I 2024-04-16 09:46:36,486] A new study created in memory with name: no-name-e309d9d9-ae5e-406e-8298-66b6f58ae171




* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6309148160341665], datetime_start=datetime.datetime(2024, 4, 16, 9, 46, 36, 370455), datetime_complete=datetime.datetime(2024, 4, 16, 9, 46, 36, 473375), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6309148160341665


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2472470987338797
Fold 2 IBS: 0.23203988176202153
Fold 3 IBS: 0.22898186794169834
Fold 4 IBS: 0.24197476878512575
Fold 5 IBS: 0.22939559108577243
[I 2024-04-16 09:46:36,676] Trial 0 finished with value: 0.23592784166169953 and parameters: {}. Best is trial 0 with value: 0.23592784166169953.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784166169953], datetime_start=datetime.datetime(2024, 4, 16, 9, 46, 36, 524752), datetime_complete=datetime.datetime(2024, 4, 16, 9, 46, 36, 676417), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784166169953


In [98]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [99]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.631
train_ibs:  0.236


#### Test

In [100]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [101]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.534


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [102]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [103]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 09:46:36,844] A new study created in memory with name: no-name-721b8728-8fc2-4e6c-98bf-fc3710ca8c4d


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6920152091254753


[I 2024-04-16 09:46:37,255] A new study created in memory with name: no-name-0bc914cf-0729-4158-a53b-b96b5d682f07


Fold 5 C-index: 0.7124463519313304
[I 2024-04-16 09:46:37,250] Trial 0 finished with value: 0.6827438669030348 and parameters: {}. Best is trial 0 with value: 0.6827438669030348.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6827438669030348], datetime_start=datetime.datetime(2024, 4, 16, 9, 46, 36, 893068), datetime_complete=datetime.datetime(2024, 4, 16, 9, 46, 37, 250048), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6827438669030348


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2626935744757735
Fold 2 IBS: 0.17625232710995545
Fold 3 IBS: 0.19835796742294537
Fold 4 IBS: 0.24640861437214787
Fold 5 IBS: 0.1894678752019021
[I 2024-04-16 09:46:37,607] Trial 0 finished with value: 0.21463607171654484 and parameters: {}. Best is trial 0 with value: 0.21463607171654484.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21463607171654484], datetime_start=datetime.datetime(2024, 4, 16, 9, 46, 37, 298128), datetime_complete=datetime.datetime(2024, 4, 16, 9, 46, 37, 606991), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21463607171654484


In [104]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [105]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.683
train_ibs:  0.215


#### Test

In [106]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [107]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.507


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.282


In [108]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [109]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 09:46:37,876] A new study created in memory with name: no-name-fd7dc929-3882-41ef-816c-feb606a22ac9


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:38,207] Trial 0 finished with value: 0.6843626922755115 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6843626922755115.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:38,559] Trial 1 finished with value: 0.6836022360017473 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6843626922755115.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:38,881] Trial 2 finished with value: 0.6538150019591941 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:45,344] Trial 24 finished with value: 0.6843626922755115 and parameters: {'l1_ratio': 0.7602371709740536}. Best is trial 0 with value: 0.6843626922755115.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:45,698] Trial 25 finished with value: 0.6836022360017473 and parameters: {'l1_ratio': 0.5833004187842314}. Best is trial 0 with value: 0.6843626922755115.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:46,085] Trial 26 finished with value: 0.6843626922755115 and parameters: {'l1_ratio': 0.683379796474269}.

Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:53,504] Trial 48 finished with value: 0.6843626922755115 and parameters: {'l1_ratio': 0.5683872043393509}. Best is trial 0 with value: 0.6843626922755115.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.7124463519313304
[I 2024-04-16 09:46:53,873] Trial 49 finished with value: 0.6827438669030348 and parameters: {'l1_ratio': 0.9567711086913276}. Best is trial 0 with value: 0.6843626922755115.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7106382978723405
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.7167381974248928
[I 2024-04-16 09:46:54,346] Trial 50 finished with value: 0.6843626922755115 and parameters: {'l1_ratio': 0.7432829682775339}. Best is trial 0 with value: 0.6843626922755115.
Fold 1 C-index: 0.57

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.26257162940250955
Fold 2 IBS: 0.17633924242670115
Fold 3 IBS: 0.19822344706060782
Fold 4 IBS: 0.24637305627894682
Fold 5 IBS: 0.18905968865715259
[I 2024-04-16 09:47:13,174] Trial 0 finished with value: 0.2145134127651836 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.2145134127651836.
Fold 1 IBS: 0.26237731476072124
Fold 2 IBS: 0.17627548359861686
Fold 3 IBS: 0.19807053093982024
Fold 4 IBS: 0.24605840774168466
Fold 5 IBS: 0.18883411697939656
[I 2024-04-16 09:47:13,634] Trial 1 finished with value: 0.2143231708040479 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.2143231708040479.
Fold 1 IBS: 0.26237386642247196
Fold 2 IBS: 0.17620383273044332
Fold 3 IBS: 0.22875723206890197
Fold 4 IBS: 0.2460704170061998
Fold 5 IBS: 0.18882164380537048
[I 2024-04-16 09:47:14,039] Trial 2 finished with value: 0.2204453984066775 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 1 with value: 0.214323170804047

Fold 1 IBS: 0.2623065776975309
Fold 2 IBS: 0.22771743121877502
Fold 3 IBS: 0.22880803314680423
Fold 4 IBS: 0.23804473214879163
Fold 5 IBS: 0.22635839546653805
[I 2024-04-16 09:47:24,545] Trial 25 finished with value: 0.23664703393568795 and parameters: {'l1_ratio': 0.15838797155583242}. Best is trial 1 with value: 0.2143231708040479.
Fold 1 IBS: 0.2623009993332655
Fold 2 IBS: 0.17622672114260704
Fold 3 IBS: 0.19812409407550668
Fold 4 IBS: 0.24611110866082173
Fold 5 IBS: 0.18883518257861268
[I 2024-04-16 09:47:24,988] Trial 26 finished with value: 0.21431962115816275 and parameters: {'l1_ratio': 0.31870248699447157}. Best is trial 26 with value: 0.21431962115816275.
Fold 1 IBS: 0.26239953416003275
Fold 2 IBS: 0.176282070497932
Fold 3 IBS: 0.19806101652311164
Fold 4 IBS: 0.2460708408061116
Fold 5 IBS: 0.18883071508438132
[I 2024-04-16 09:47:25,443] Trial 27 finished with value: 0.21432883541431388 and parameters: {'l1_ratio': 0.31149345374565823}. Best is trial 26 with value: 0.214319621

Fold 4 IBS: 0.2459825779240315
Fold 5 IBS: 0.18882513676069446
[I 2024-04-16 09:47:34,794] Trial 50 finished with value: 0.22041390883740436 and parameters: {'l1_ratio': 0.2330625599543072}. Best is trial 32 with value: 0.2143058730064582.
Fold 1 IBS: 0.2623859572633629
Fold 2 IBS: 0.17627844132590192
Fold 3 IBS: 0.19806615177383452
Fold 4 IBS: 0.24606570142281473
Fold 5 IBS: 0.18883773320582636
[I 2024-04-16 09:47:35,241] Trial 51 finished with value: 0.21432679699834808 and parameters: {'l1_ratio': 0.28707431039828885}. Best is trial 32 with value: 0.2143058730064582.
Fold 1 IBS: 0.2622418083206451
Fold 2 IBS: 0.22819689871071516
Fold 3 IBS: 0.2288241611255437
Fold 4 IBS: 0.2384579244113865
Fold 5 IBS: 0.22669399356844042
[I 2024-04-16 09:47:35,447] Trial 52 finished with value: 0.23688295722734617 and parameters: {'l1_ratio': 0.1404992778339893}. Best is trial 32 with value: 0.2143058730064582.
Fold 1 IBS: 0.2623253164824203
Fold 2 IBS: 0.17624295609223664
Fold 3 IBS: 0.198127796219

Fold 1 IBS: 0.2623075377468284
Fold 2 IBS: 0.17624816741614854
Fold 3 IBS: 0.22876227158276394
Fold 4 IBS: 0.24601075283931992
Fold 5 IBS: 0.18881592231016656
[I 2024-04-16 09:47:45,093] Trial 75 finished with value: 0.22042893037904548 and parameters: {'l1_ratio': 0.21877076880955348}. Best is trial 71 with value: 0.21430408386786345.
Fold 1 IBS: 0.26238165234868044
Fold 2 IBS: 0.17626360950860542
Fold 3 IBS: 0.1980492988603829
Fold 4 IBS: 0.24617008805051238
Fold 5 IBS: 0.18884853158303821
[I 2024-04-16 09:47:45,550] Trial 76 finished with value: 0.21434263607024384 and parameters: {'l1_ratio': 0.3557517419143058}. Best is trial 71 with value: 0.21430408386786345.
Fold 1 IBS: 0.26226999857264344
Fold 2 IBS: 0.17622334636572434
Fold 3 IBS: 0.22874420184293884
Fold 4 IBS: 0.2461012750413376
Fold 5 IBS: 0.18882727787725043
[I 2024-04-16 09:47:46,019] Trial 77 finished with value: 0.22043321993997894 and parameters: {'l1_ratio': 0.25086758887372973}. Best is trial 71 with value: 0.214304

In [110]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [111]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.684
train_ibs:  0.214


#### Test

In [112]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [113]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.506


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.32614159377093416)

test_ibs:  0.282


In [114]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [115]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 09:47:57,714] A new study created in memory with name: no-name-506e6cb8-9e1b-4b72-b750-49ae36620c1f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6972111553784861
Fold 2 C-index: 0.7558139534883721
Fold 3 C-index: 0.6382978723404256
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6824034334763949
[I 2024-04-16 09:48:02,714] Trial 0 finished with value: 0.7015133437732377 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7015133437732377.
Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.7276595744680852
Fold 4 C-index: 0.779467680608365
Fold 5 C-index: 0.6909871244635193
[I 2024-04-16 09:48:07,264] Trial 1 finished with value: 0.715218169153616 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7870722433460076
Fold 5 C-index: 0.6995708154506438
[I 2024-04-16 09:49:09,780] Trial 15 finished with value: 0.7085249151159188 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 8, 'n_estimators': 338, 'oob_score': False, 'max_samples': 0.8070084084612321, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08673913474661687, 'warm_start': False}. Best is trial 1 with value: 0.715218169153616.
Fold 1 C-index: 0.6573705179282868
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.7319148936170212
Fold 4 C-index: 0.7756653992395437
Fold 5 C-index: 0.6866952789699571
[I 2024-04-16 09:49:12,758] Trial 16 finished with value: 0.7183912334548377 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 13, 'min_samples_leaf': 10, 'max_depth': 20, 'n_estimators': 237, 'oob_score': False, 'max_samples': 0.6501934815344157, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.000100830065

Fold 1 C-index: 0.649402390438247
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.7756653992395437
Fold 5 C-index: 0.703862660944206
[I 2024-04-16 09:49:47,847] Trial 30 finished with value: 0.7161275052654187 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 15, 'n_estimators': 303, 'oob_score': False, 'max_samples': 0.9063670455515496, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.09013372355598072, 'warm_start': False}. Best is trial 16 with value: 0.7183912334548377.
Fold 1 C-index: 0.6573705179282868
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.7148936170212766
Fold 4 C-index: 0.7718631178707225
Fold 5 C-index: 0.6866952789699571
[I 2024-04-16 09:49:55,854] Trial 31 finished with value: 0.7111257466681262 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 16, 'n_estimators': 317, 'oob_score': False, 'max_samples': 0.8980407898

...

Fold 5 IBS: 0.2087860846359583
[I 2024-04-16 09:55:13,371] Trial 31 finished with value: 0.2062433096017649 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 15, 'min_samples_leaf': 5, 'max_depth': 3, 'n_estimators': 246, 'oob_score': True, 'max_samples': 0.6079288251856192, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.043283773765183414}. Best is trial 18 with value: 0.20528919347758562.
Fold 1 IBS: 0.2297035767904694
Fold 2 IBS: 0.18353852060302428
Fold 3 IBS: 0.19854054014764616
Fold 4 IBS: 0.20605499505040847
Fold 5 IBS: 0.20598946772047116
[I 2024-04-16 09:55:15,985] Trial 32 finished with value: 0.20476542006240392 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 6, 'max_depth': 3, 'n_estimators': 168, 'oob_score': True, 'max_samples': 0.6522520875850281, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.023069227012922855}. Best is trial 32 with value: 0.20476542006240392.
Fold 1 IBS: 0.22593546080444016
Fold 2 IBS: 0.183

Fold 1 IBS: 0.22547637004046184
Fold 2 IBS: 0.18496551370930772
Fold 3 IBS: 0.20478408769029746
Fold 4 IBS: 0.21368791701531417
Fold 5 IBS: 0.21306360231478774
[I 2024-04-16 09:55:48,458] Trial 47 finished with value: 0.20839549815403377 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 19, 'min_samples_leaf': 9, 'max_depth': 6, 'n_estimators': 78, 'oob_score': True, 'max_samples': 0.43491671167283275, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.07153350270189084}. Best is trial 33 with value: 0.202541656814584.
Fold 1 IBS: 0.24590405124273745
Fold 2 IBS: 0.23203729345433474
Fold 3 IBS: 0.2306508774403552
Fold 4 IBS: 0.24143250050767692
Fold 5 IBS: 0.22988062785342173
[I 2024-04-16 09:55:49,323] Trial 48 finished with value: 0.2359810700997052 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 4, 'n_estimators': 35, 'oob_score': True, 'max_samples': 0.7488226031933044, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.22738980486047694
Fold 2 IBS: 0.18424498258675306
Fold 3 IBS: 0.20401800243828214
Fold 4 IBS: 0.20439775352080095
Fold 5 IBS: 0.20383642835164115
[I 2024-04-16 09:56:31,899] Trial 63 finished with value: 0.20477739435159084 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 13, 'max_depth': 13, 'n_estimators': 182, 'oob_score': True, 'max_samples': 0.9262735412221665, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.08375003843429962}. Best is trial 57 with value: 0.2023298174608712.
Fold 1 IBS: 0.2297357114083422
Fold 2 IBS: 0.18475208112109387
Fold 3 IBS: 0.21076456987331543
Fold 4 IBS: 0.20644128003060727
Fold 5 IBS: 0.2085911701115838
[I 2024-04-16 09:56:34,102] Trial 64 finished with value: 0.20805696250898853 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 11, 'n_estimators': 152, 'oob_score': True, 'max_samples': 0.8449961874170873, 'max_features': 'log2', 'min_weight_fractio

Fold 5 IBS: 0.20778331062924044
[I 2024-04-16 09:57:23,175] Trial 78 finished with value: 0.21940336484536585 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 7, 'n_estimators': 143, 'oob_score': True, 'max_samples': 0.9696883436320323, 'max_features': None, 'min_weight_fraction_leaf': 0.03521042067680326}. Best is trial 73 with value: 0.20159715567876108.
Fold 1 IBS: 0.23036951675743086
Fold 2 IBS: 0.18464849451895313
Fold 3 IBS: 0.2043713140769145
Fold 4 IBS: 0.20232000830215705
Fold 5 IBS: 0.2000574478630733
[I 2024-04-16 09:57:27,694] Trial 79 finished with value: 0.20435335630370574 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 264, 'oob_score': True, 'max_samples': 0.921109290734505, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.01634003279013976}. Best is trial 73 with value: 0.20159715567876108.
Fold 1 IBS: 0.22952283595294182
Fold 2 IBS: 0.1840

Fold 1 IBS: 0.22661154321542654
Fold 2 IBS: 0.1826314318857676
Fold 3 IBS: 0.20258564093938022
Fold 4 IBS: 0.20038355387682857
Fold 5 IBS: 0.201005200159819
[I 2024-04-16 09:58:05,765] Trial 94 finished with value: 0.2026434740154444 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 16, 'min_samples_leaf': 12, 'max_depth': 13, 'n_estimators': 150, 'oob_score': False, 'max_samples': 0.9930073589533785, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.008971163622332974}. Best is trial 73 with value: 0.20159715567876108.
Fold 1 IBS: 0.2226668884661819
Fold 2 IBS: 0.18494614156725506
Fold 3 IBS: 0.204837192196963
Fold 4 IBS: 0.20461549662877815
Fold 5 IBS: 0.2036474473766526
[I 2024-04-16 09:58:07,653] Trial 95 finished with value: 0.20414263324716614 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 15, 'max_depth': 14, 'n_estimators': 123, 'oob_score': False, 'max_samples': 0.9975077981339631, 'max_features': 'sqrt', 'min_weight_fraction_le

In [116]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [117]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.797
train_ibs:  0.202


#### Test

In [118]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [119]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=18, max_features='log2', max_leaf_nodes=6,
                     max_samples=0.7275622802375871, min_samples_split=17,
                     min_weight_fraction_leaf=0.010224090801480467,
                     n_estimators=281, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.517


RandomSurvivalForest(max_depth=10, max_leaf_nodes=20,
                     max_samples=0.9280035754058158, min_samples_leaf=11,
                     min_samples_split=13,
                     min_weight_fraction_leaf=0.0918192183935895,
                     n_estimators=165, oob_score=True, random_state=123)

test_ibs:  0.261


In [120]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [121]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [122]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 09:58:25,466] A new study created in memory with name: no-name-60b2bac2-2e21-48b1-8745-d7d5039651c5


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7751937984496124
Fold 3 C-index: 0.7787234042553192
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.776824034334764
[I 2024-04-16 09:58:26,661] Trial 0 finished with value: 0.736891888811905 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.736891888811905.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 09:58:29,599] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. 

Fold 5 C-index: 0.7124463519313304
[I 2024-04-16 09:59:08,965] Trial 15 finished with value: 0.7059016436492067 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7524534888374881.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.7790697674418605
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.6749049429657795
Fold 5 C-index: 0.6223175965665236
[I 2024-04-16 09:59:11,488] Trial 16 finished with value: 0.6633435677778114 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7524534888374881.
Fold 1 

Fold 1 C-index: 0.549800796812749
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7574468085106383
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.6781115879828327
[I 2024-04-16 09:59:40,247] Trial 30 finished with value: 0.6974166967388813 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.23596047192138714, 'min_weight_fraction_leaf': 0.09139810961881356}. Best is trial 12 with value: 0.7524534888374881.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7674418604651163
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.776824034334764
[I 2024-04-16 09:59:42,104] Trial 31 finished with value: 0.7378767980988239 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 11, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 453, 'oob_score': False, 'warm_start': True, 'max_feature

...

Fold 1 IBS: 0.242353362091815
Fold 2 IBS: 0.21339799384020264
Fold 3 IBS: 0.21726638914512997
Fold 4 IBS: 0.23267246744838713
Fold 5 IBS: 0.2212980602481738
[I 2024-04-16 10:03:34,343] Trial 30 finished with value: 0.22539765455474173 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 457, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.20961630150441302.
Fold 1 IBS: 0.23674198115352363
Fold 2 IBS: 0.19518122847366612
Fold 3 IBS: 0.2050120018943178
Fold 4 IBS: 0.22046781019161954
Fold 5 IBS: 0.21215749826372995
[I 2024-04-16 10:03:37,962] Trial 31 finished with value: 0.2139121039953714 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7

Fold 1 IBS: 0.2347053268942383
Fold 2 IBS: 0.1938225355692269
Fold 3 IBS: 0.19988939553972823
Fold 4 IBS: 0.22031520927619141
Fold 5 IBS: 0.21043236509727797
[I 2024-04-16 10:04:14,811] Trial 45 finished with value: 0.2118329664753326 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 8, 'n_estimators': 284, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7161510423088122, 'min_weight_fraction_leaf': 0.05492743841376213}. Best is trial 35 with value: 0.2064799261510724.
Fold 1 IBS: 0.23315772457878367
Fold 2 IBS: 0.19283289362929562
Fold 3 IBS: 0.2009501574410602
Fold 4 IBS: 0.22027594986911345
Fold 5 IBS: 0.20896264084334623
[I 2024-04-16 10:04:17,711] Trial 46 finished with value: 0.2112358732723198 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 314, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.77

Fold 1 IBS: 0.23265950444998454
Fold 2 IBS: 0.18636127453099866
Fold 3 IBS: 0.19993958938359596
Fold 4 IBS: 0.2162253571045426
Fold 5 IBS: 0.20381100578791037
[I 2024-04-16 10:04:50,231] Trial 60 finished with value: 0.20779934625140645 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 219, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.9208992051748564, 'min_weight_fraction_leaf': 0.08831048402058213}. Best is trial 57 with value: 0.20571691773188383.
Fold 1 IBS: 0.2310876809515107
Fold 2 IBS: 0.18774490398843385
Fold 3 IBS: 0.20242152649651662
Fold 4 IBS: 0.21366529710198845
Fold 5 IBS: 0.20379492815843792
[I 2024-04-16 10:04:52,559] Trial 61 finished with value: 0.20774286733937747 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 232, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.

Fold 1 IBS: 0.22515802372915686
Fold 2 IBS: 0.19075202603689245
Fold 3 IBS: 0.20939834429255869
Fold 4 IBS: 0.21245938135827597
Fold 5 IBS: 0.20851851328898718
[I 2024-04-16 10:05:29,514] Trial 75 finished with value: 0.2092572577411742 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 8, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 250, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.7564438927738872, 'min_weight_fraction_leaf': 0.12828483764121068}. Best is trial 57 with value: 0.20571691773188383.
Fold 1 IBS: 0.22744289060382553
Fold 2 IBS: 0.19075206728432154
Fold 3 IBS: 0.20554651061472592
Fold 4 IBS: 0.21217294361124195
Fold 5 IBS: 0.20891216606780766
[I 2024-04-16 10:05:31,993] Trial 76 finished with value: 0.2089653156363845 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 273, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.

Fold 1 IBS: 0.23162104883048065
Fold 2 IBS: 0.18920181466408656
Fold 3 IBS: 0.1946949770737152
Fold 4 IBS: 0.21631250088639659
Fold 5 IBS: 0.20244287119552642
[I 2024-04-16 10:06:10,513] Trial 90 finished with value: 0.2068546425300411 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 10, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 375, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7134961759303595, 'min_weight_fraction_leaf': 0.009412107712000094}. Best is trial 57 with value: 0.20571691773188383.
Fold 1 IBS: 0.23181164849341912
Fold 2 IBS: 0.1893769722084939
Fold 3 IBS: 0.19447302134656916
Fold 4 IBS: 0.2163472127256083
Fold 5 IBS: 0.20229925588198513
[I 2024-04-16 10:06:13,810] Trial 91 finished with value: 0.20686162213121512 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 10, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 370, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples':

In [123]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [124]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.798
train_ibs:  0.205


#### Test

In [125]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [126]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=18, max_features=None, max_leaf_nodes=16,
                   max_samples=0.7746492428469892, min_samples_leaf=4,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.02592476635662061,
                   n_estimators=190, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.506


ExtraSurvivalTrees(max_depth=19, max_features='log2', max_leaf_nodes=11,
                   max_samples=0.7338953025585497, min_samples_leaf=1,
                   min_samples_split=18,
                   min_weight_fraction_leaf=0.008989573072247403,
                   n_estimators=375, random_state=123, warm_start=True)

IBS: 0.25


In [127]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [128]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 10:06:43,627] A new study created in memory with name: no-name-2a3db80f-1c91-4492-a0b3-3e9b4413f1b5


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:07:02,476] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:07:12,540] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:12:27,483] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.689211725925309.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:13:12,601] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:21:47,311] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 12 with value: 0.689211725925309.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 10:22:09,592] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 5 IBS: 0.22939559304809248
[I 2024-04-16 11:05:28,411] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.20419816136861105, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 1.5125790196698774, 'min_weight_fraction_leaf': 0.21953014805517473, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.9659902853562541, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 2}. Best is trial 41 with value: 0.2344232217550511.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 11:05:56,796] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2887765610954624, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.23996644019536584, 'n_estimators': 3

Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 11:11:51,428] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.5392724474754766, 'learning_rate': 0.01461816521127875, 'dropout_rate': 0.22076051954616932, 'n_estimators': 377, 'criterion': 'squared_error', 'ccp_alpha': 0.8099144041738244, 'min_weight_fraction_leaf': 0.03518122515344503, 'max_features': 'sqrt', 'min_impurity_decrease': 1.1843050264198749e-07, 'validation_fraction': 0.9678192460592685, 'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 63 with value: 0.233993735199733.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 11:12:26,218] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.680855021230625, 'learning_rate': 0.004919883070534993, 'dropout_rate': 0.17168408254323692, 'n_estimators': 420

Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 11:15:13,698] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.5357913465602108, 'learning_rate': 0.052009701795490706, 'dropout_rate': 0.19884799526452568, 'n_estimators': 126, 'criterion': 'squared_error', 'ccp_alpha': 0.5891837005828349, 'min_weight_fraction_leaf': 0.02046465024104037, 'max_features': 1, 'min_impurity_decrease': 0.00801697775600995, 'validation_fraction': 0.8634632458479912, 'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 9, 'max_depth': 13}. Best is trial 72 with value: 0.23059942917612353.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 11:15:38,878] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9721672658830589, 'learning_rate': 0.0869328686638194, 'dropout_rate': 0.13115268850819523, 'n_estimators': 358, 'crit

Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 11:20:11,875] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.49209825028823895, 'learning_rate': 0.010473050195075363, 'dropout_rate': 0.20588321060682593, 'n_estimators': 423, 'criterion': 'squared_error', 'ccp_alpha': 1.099942037302291, 'min_weight_fraction_leaf': 0.08127692641673222, 'max_features': 'auto', 'min_impurity_decrease': 1.5480811100354597e-07, 'validation_fraction': 0.696357383667876, 'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 4, 'max_depth': 1}. Best is trial 82 with value: 0.2277344544728534.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-16 11:20:32,932] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.995129754492818, 'learning_rate': 0.04745413618139034, 'dropout_rate': 0.35768717188704796, 'n_estimators': 367, 

Fold 5 IBS: 0.22576583126917327
[I 2024-04-16 11:25:49,605] Trial 99 finished with value: 0.2297576529640855 and parameters: {'subsample': 0.8869168213912387, 'learning_rate': 0.03566645000969104, 'dropout_rate': 0.10216550214025771, 'n_estimators': 447, 'criterion': 'squared_error', 'ccp_alpha': 0.004367203675973822, 'min_weight_fraction_leaf': 0.1402070137569511, 'max_features': 'sqrt', 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.9870731812554687, 'min_samples_split': 11, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 1}. Best is trial 82 with value: 0.2277344544728534.


* Best trial for IBS: 
 FrozenTrial(number=82, state=TrialState.COMPLETE, values=[0.2277344544728534], datetime_start=datetime.datetime(2024, 4, 16, 11, 16, 21, 347118), datetime_complete=datetime.datetime(2024, 4, 16, 11, 16, 55, 57565), params={'subsample': 0.6994793948584171, 'learning_rate': 0.06401898097796113, 'dropout_rate': 0.19156645482498222, 'n_estimators': 426, 'c

In [129]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [130]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.714
train_ibs:  0.228


#### Test

In [131]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [132]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.05874457198222169,
                                 dropout_rate=0.8142690261508861,
                                 learning_rate=0.02998154803834563, max_depth=4,
                                 max_features=0.1, max_leaf_nodes=5,
                                 min_impurity_decrease=4.899193303970509e-05,
                                 min_samples_leaf=13, min_samples_split=12,
                                 min_weight_fraction_leaf=0.3150248435971541,
                                 n_estimators=161, random_state=123,
                                 subsample=0.5167163717577359,
                                 validation_fraction=0.8336552353722929)

C-index score: 0.544


GradientBoostingSurvivalAnalysis(ccp_alpha=0.0177990205062633,
                                 criterion='squared_error',
                                 dropout_rate=0.19156645482498222,
                                 learning_rate=0.06401898097796113, max_depth=2,
                                 max_features='sqrt', max_leaf_nodes=17,
                                 min_impurity_decrease=2.601503817654743e-07,
                                 min_samples_leaf=6, min_samples_split=3,
                                 min_weight_fraction_leaf=0.08269791710039148,
                                 n_estimators=426, random_state=123,
                                 subsample=0.6994793948584171,
                                 validation_fraction=0.6514317760469892)

IBS: 0.229


In [133]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [134]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [135]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 11:25:57,290] A new study created in memory with name: no-name-54480853-85a6-4544-8580-40d03b6a55ae


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.748062015503876
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6159695817490495
Fold 5 C-index: 0.6008583690987125
[I 2024-04-16 11:25:57,895] Trial 0 finished with value: 0.6184658291608083 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6184658291608083.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6159695817490495
Fold 5 C-index: 0.6008583690987125
[I 2024-04-16 11:26:02,711] Trial 1 finished with value: 0.6192410229592579 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6192410229592579.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.6085106382978723
Fold 4 

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.5779467680608364
Fold 5 C-index: 0.6051502145922747
[I 2024-04-16 11:26:50,747] Trial 19 finished with value: 0.6363246165543702 and parameters: {'subsample': 0.19720963658850754, 'dropout_rate': 0.6165615625273371, 'n_estimators': 400, 'learning_rate': 0.09107738339755381}. Best is trial 16 with value: 0.639674620792736.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.6137339055793991
[I 2024-04-16 11:26:52,354] Trial 20 finished with value: 0.6315697376565172 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.7906239277194107, 'n_estimators': 258, 'learning_rate': 0.06551866052750378}. Best is trial 16 with value: 0.639674620792736.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7558139534883721
Fold 3 C-index: 0.6936170212765957
Fol

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.5872340425531914
Fold 4 C-index: 0.5893536121673004
Fold 5 C-index: 0.6008583690987125
[I 2024-04-16 11:27:40,828] Trial 38 finished with value: 0.6205745996185306 and parameters: {'subsample': 0.6555428090282759, 'dropout_rate': 0.15265508330268462, 'n_estimators': 439, 'learning_rate': 0.05809962418833657}. Best is trial 33 with value: 0.6415435228958922.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7558139534883721
Fold 3 C-index: 0.6510638297872341
Fold 4 C-index: 0.5779467680608364
Fold 5 C-index: 0.6180257510729614
[I 2024-04-16 11:27:44,300] Trial 39 finished with value: 0.6337174708404466 and parameters: {'subsample': 0.3117059866773502, 'dropout_rate': 0.5217112798506811, 'n_estimators': 388, 'learning_rate': 0.06892741183938003}. Best is trial 33 with value: 0.6415435228958922.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.6382978723404256


...

Fold 3 IBS: 0.23176649970092905
Fold 4 IBS: 0.22914748233649584
Fold 5 IBS: 0.22201402626583885
[I 2024-04-16 11:31:51,038] Trial 19 finished with value: 0.225801809161084 and parameters: {'subsample': 0.94616158758761, 'dropout_rate': 0.22827203020188386, 'n_estimators': 70, 'learning_rate': 0.08697733590757546}. Best is trial 19 with value: 0.225801809161084.
Fold 1 IBS: 0.24959950048357138
Fold 2 IBS: 0.19954737342877818
Fold 3 IBS: 0.2300253764336497
Fold 4 IBS: 0.23015441638199688
Fold 5 IBS: 0.22218063467579216
[I 2024-04-16 11:31:51,315] Trial 20 finished with value: 0.22630146028075765 and parameters: {'subsample': 0.8915988185901454, 'dropout_rate': 0.7933084651006226, 'n_estimators': 54, 'learning_rate': 0.0817175642960023}. Best is trial 19 with value: 0.225801809161084.
Fold 1 IBS: 0.25343805486117815
Fold 2 IBS: 0.1929792488703679
Fold 3 IBS: 0.23126028185408112
Fold 4 IBS: 0.2295679856672015
Fold 5 IBS: 0.22192783273263939
[I 2024-04-16 11:31:51,614] Trial 21 finished wit

Fold 1 IBS: 0.25319952780709715
Fold 2 IBS: 0.19447065295499547
Fold 3 IBS: 0.2308832935467258
Fold 4 IBS: 0.23130284304424584
Fold 5 IBS: 0.2219474736234986
[I 2024-04-16 11:32:02,835] Trial 39 finished with value: 0.22636075819531257 and parameters: {'subsample': 0.8264537565050025, 'dropout_rate': 0.5138173926600543, 'n_estimators': 199, 'learning_rate': 0.026783213087327076}. Best is trial 19 with value: 0.225801809161084.
Fold 1 IBS: 0.35748458583075876
Fold 2 IBS: 0.1574368697533429
Fold 3 IBS: 0.28413193107336593
Fold 4 IBS: 0.29041667511576114
Fold 5 IBS: 0.2921425104485691
[I 2024-04-16 11:32:04,870] Trial 40 finished with value: 0.27632251444435957 and parameters: {'subsample': 0.8890339254719992, 'dropout_rate': 0.280971539366882, 'n_estimators': 289, 'learning_rate': 0.0996566606976679}. Best is trial 19 with value: 0.225801809161084.
Fold 1 IBS: 0.2533457475769324
Fold 2 IBS: 0.19315732935382318
Fold 3 IBS: 0.23104002496179532
Fold 4 IBS: 0.2293951872360855
Fold 5 IBS: 0.2

Fold 2 IBS: 0.18716533330408597
Fold 3 IBS: 0.23303197607358758
Fold 4 IBS: 0.23398477041757762
Fold 5 IBS: 0.22216692496562387
[I 2024-04-16 11:32:16,983] Trial 58 finished with value: 0.22714081357881727 and parameters: {'subsample': 0.8049034694205583, 'dropout_rate': 0.5787791443289121, 'n_estimators': 154, 'learning_rate': 0.043247386812275496}. Best is trial 41 with value: 0.2257740928945569.
Fold 1 IBS: 0.24520739805611824
Fold 2 IBS: 0.212288887450923
Fold 3 IBS: 0.22892286779651505
Fold 4 IBS: 0.234841528625502
Fold 5 IBS: 0.22406946857168855
[I 2024-04-16 11:32:17,347] Trial 59 finished with value: 0.2290660301001494 and parameters: {'subsample': 0.8394135718238601, 'dropout_rate': 0.8497981022720253, 'n_estimators': 79, 'learning_rate': 0.03241534540850621}. Best is trial 41 with value: 0.2257740928945569.
Fold 1 IBS: 0.28252826808016807
Fold 2 IBS: 0.16820900927725183
Fold 3 IBS: 0.22493067580151604
Fold 4 IBS: 0.2564161095583284
Fold 5 IBS: 0.22837098545596912
[I 2024-04-1

Fold 5 IBS: 0.22195101233402054
[I 2024-04-16 11:32:28,022] Trial 77 finished with value: 0.2258673161226314 and parameters: {'subsample': 0.9241806819491492, 'dropout_rate': 0.6015827229072146, 'n_estimators': 96, 'learning_rate': 0.05340669166408334}. Best is trial 41 with value: 0.2257740928945569.
Fold 1 IBS: 0.2502303777379061
Fold 2 IBS: 0.19871411116671306
Fold 3 IBS: 0.2301958168448245
Fold 4 IBS: 0.23109356781896448
Fold 5 IBS: 0.2220986101175287
[I 2024-04-16 11:32:28,477] Trial 78 finished with value: 0.22646649673718736 and parameters: {'subsample': 0.8563848810472314, 'dropout_rate': 0.6027083599398141, 'n_estimators': 98, 'learning_rate': 0.047078364671080265}. Best is trial 41 with value: 0.2257740928945569.
Fold 1 IBS: 0.27416643596802237
Fold 2 IBS: 0.1746513042353999
Fold 3 IBS: 0.23725344964183798
Fold 4 IBS: 0.23344106600037298
Fold 5 IBS: 0.22537565707734725
[I 2024-04-16 11:32:29,347] Trial 79 finished with value: 0.2289775825845961 and parameters: {'subsample': 0

Fold 3 IBS: 0.22944653432708567
Fold 4 IBS: 0.23063827375154522
Fold 5 IBS: 0.2229432534166449
[I 2024-04-16 11:32:38,916] Trial 97 finished with value: 0.2270694961444221 and parameters: {'subsample': 0.9767455260315752, 'dropout_rate': 0.6488691992826511, 'n_estimators': 87, 'learning_rate': 0.04071453518360497}. Best is trial 85 with value: 0.22569422644217568.
Fold 1 IBS: 0.24480764542120276
Fold 2 IBS: 0.21626586987434746
Fold 3 IBS: 0.2272316015685666
Fold 4 IBS: 0.23945647308726237
Fold 5 IBS: 0.2251722183706185
[I 2024-04-16 11:32:39,152] Trial 98 finished with value: 0.2305867616643995 and parameters: {'subsample': 0.5666236381817311, 'dropout_rate': 0.6788633161910144, 'n_estimators': 36, 'learning_rate': 0.051501923007209446}. Best is trial 85 with value: 0.22569422644217568.
Fold 1 IBS: 0.2944680130865383
Fold 2 IBS: 0.1628625094014068
Fold 3 IBS: 0.24556046949396654
Fold 4 IBS: 0.24468301864678557
Fold 5 IBS: 0.23483960337036835
[I 2024-04-16 11:32:40,304] Trial 99 finishe

In [136]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [137]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.646
train_ibs:  0.226


#### Test

In [138]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [139]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5065802886056728,
                                              learning_rate=0.08860071779281121,
                                              n_estimators=417,
                                              random_state=123,
                                              subsample=0.10200972368178224)

C-index score: 0.54


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7450922639673777,
                                              learning_rate=0.05183366352011241,
                                              n_estimators=107,
                                              random_state=123,
                                              subsample=0.9715715213181028)

IBS: 0.233


In [140]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [141]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.798,1.0
Randomsurvivalforest,0.797,2.0
GradientBoosting,0.714,3.0
CoxElastic,0.684,4.0
CoxLasso,0.683,5.0
CoxPH,0.681,6.0
ComponentwiseGradientBoosting,0.646,7.0
CoxRidge,0.631,8.0


In [142]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.202,1.0
ExtraSurvivalTrees,0.205,2.0
CoxElastic,0.214,3.0
CoxLasso,0.215,4.0
CoxPH,0.216,5.0
ComponentwiseGradientBoosting,0.226,6.0
GradientBoosting,0.228,7.0
CoxRidge,0.236,8.0


In [143]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.544,1.0
ComponentwiseGradientBoosting,0.540,2.0
CoxRidge,0.534,3.0
Randomsurvivalforest,0.517,4.0
CoxPH,0.508,5.0
CoxLasso,0.507,6.0
CoxElastic,0.506,7.5
ExtraSurvivalTrees,0.506,7.5


In [144]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ComponentwiseGradientBoosting,0.233,3.0
ExtraSurvivalTrees,0.250,4.0
Randomsurvivalforest,0.261,5.0
CoxLasso,0.282,6.5
CoxElastic,0.282,6.5
CoxPH,0.283,8.0


In [145]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/minmax/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_minmax_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [146]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-16
